# DeepSeek Attention Architecture (MLA + Decoupled RoPE)

# Building the MLA Module from Scratch

In [1]:
# Purpose:
# - Understand DeepSeek's MLA architecture
# - Learn how MLA compresses KV Cache memory
# - See the "Compress → Store → Decompress" workflow
#
# Key Idea:
#
# Standard MHA:
#     Cache full K and V matrices
#
# MQA:
#     Share K and V across heads
#
# MLA (DeepSeek):
#     Compress K and V into a small latent representation
#     Cache only the compressed latent vector
#     Reconstruct K and V when needed
#
# This dramatically reduces KV Cache memory while preserving attention quality.
# ============================================================

# ============================================================
# Import Required Libraries
# ============================================================

import torch  # PyTorch tensor library
import torch.nn as nn  # Neural network modules


# ============================================================
# Multi-Head Latent Attention (MLA)
# ============================================================

class MultiHeadLatentAttention(nn.Module):
    """
    Implementation of Multi-Head Latent Attention (MLA).

    DeepSeek's main idea:

    Instead of caching:
        K, V

    Cache:
        Compressed latent vector c_kv

    Later reconstruct:
        c_kv -> K
        c_kv -> V

    This significantly reduces KV Cache memory.
    """

    # --------------------------------------------------------
    # Constructor
    # --------------------------------------------------------
    def __init__(
        self,
        d_model,
        num_heads,
        d_latent,
        dropout=0.0
    ):

        # Initialize parent nn.Module
        super().__init__()

        # Ensure hidden dimension is divisible by number of heads
        assert d_model % num_heads == 0, \
            "d_model must be divisible by num_heads"

        # Total embedding dimension
        #
        # Example:
        # 512
        self.d_model = d_model

        # Number of attention heads
        #
        # Example:
        # 8
        self.num_heads = num_heads

        # Dimension per head
        #
        # Example:
        # 512 / 8 = 64
        self.d_head = d_model // num_heads

        # Compressed latent dimension
        #
        # Example:
        # 128
        #
        # Must be much smaller than d_model
        self.d_latent = d_latent

        # ----------------------------------------------------
        # Query Projection
        # ----------------------------------------------------
        #
        # Queries are NOT compressed.
        #
        # Input:
        # (B, seq_len, d_model)
        #
        # Output:
        # (B, seq_len, d_model)
        self.W_q = nn.Linear(
            d_model,
            d_model
        )

        # ----------------------------------------------------
        # KV Down Projection
        # ----------------------------------------------------
        #
        # MLA Innovation
        #
        # Compress:
        #
        # d_model -> d_latent
        #
        # Example:
        #
        # 512 -> 128
        #
        # This compressed representation is what gets cached.
        self.W_dkv = nn.Linear(
            d_model,
            d_latent
        )

        # ----------------------------------------------------
        # Key Up Projection
        # ----------------------------------------------------
        #
        # Reconstruct Key matrix
        #
        # d_latent -> d_model
        #
        # Example:
        #
        # 128 -> 512
        self.W_uk = nn.Linear(
            d_latent,
            d_model
        )

        # ----------------------------------------------------
        # Value Up Projection
        # ----------------------------------------------------
        #
        # Reconstruct Value matrix
        #
        # d_latent -> d_model
        self.W_uv = nn.Linear(
            d_latent,
            d_model
        )

        # ----------------------------------------------------
        # Final Output Projection
        # ----------------------------------------------------
        self.W_o = nn.Linear(
            d_model,
            d_model
        )

        # Dropout layer
        self.dropout = nn.Dropout(dropout)

        # ----------------------------------------------------
        # Causal Mask
        # ----------------------------------------------------
        #
        # Prevents future token access.
        #
        # Shape:
        # (1,1,1024,1024)
        self.register_buffer(
            'mask',
            torch.triu(
                torch.ones(
                    1,
                    1,
                    1024,
                    1024
                ),
                diagonal=1
            ).bool()
        )

    # ========================================================
    # Forward Pass
    # ========================================================
    def forward(self, x):

        # Input shape:
        #
        # (batch_size, seq_len, d_model)
        batch_size, seq_len, _ = x.shape

        # ====================================================
        # Step 1: Query Path
        # ====================================================
        #
        # Queries remain standard.
        #
        # No compression applied.
        q = self.W_q(x)

        # Reshape into heads
        #
        # (B, seq_len, num_heads, d_head)
        q = q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_head
        )

        # Move heads dimension forward
        #
        # (B, num_heads, seq_len, d_head)
        q = q.transpose(1, 2)

        # ====================================================
        # Step 2: Compress KV Information
        # ====================================================
        #
        # MLA Core Innovation
        #
        # Compress:
        #
        # (B, seq_len, 512)
        #
        # ->
        #
        # (B, seq_len, 128)
        #
        # This latent representation is what would
        # actually be stored inside KV Cache.
        c_kv = self.W_dkv(x)

        # ====================================================
        # Step 3: Reconstruct Key Matrix
        # ====================================================
        #
        # Decompress:
        #
        # 128 -> 512
        k = self.W_uk(c_kv)

        # Split into heads
        #
        # (B, seq_len, num_heads, d_head)
        k = k.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_head
        )

        # (B, num_heads, seq_len, d_head)
        k = k.transpose(1, 2)

        # ====================================================
        # Step 4: Reconstruct Value Matrix
        # ====================================================
        #
        # Decompress:
        #
        # 128 -> 512
        v = self.W_uv(c_kv)

        # Split into heads
        v = v.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_head
        )

        # (B, num_heads, seq_len, d_head)
        v = v.transpose(1, 2)

        # ====================================================
        # Step 5: Compute Attention Scores
        # ====================================================
        #
        # Formula:
        #
        # Attention Scores
        #
        # = QKᵀ / √d_head
        #
        # Shape:
        # (B, num_heads, seq_len, seq_len)
        attn_scores = (
            q @ k.transpose(-2, -1)
        ) / (self.d_head ** 0.5)

        # ====================================================
        # Step 6: Apply Causal Mask
        # ====================================================
        #
        # Future positions receive -∞
        #
        # Softmax(-∞) = 0
        attn_scores = attn_scores.masked_fill(
            self.mask[:, :, :seq_len, :seq_len],
            float('-inf')
        )

        # ====================================================
        # Step 7: Convert Scores to Probabilities
        # ====================================================
        attn_weights = torch.softmax(
            attn_scores,
            dim=-1
        )

        # Apply dropout
        attn_weights = self.dropout(attn_weights)

        # ====================================================
        # Step 8: Compute Context Vector
        # ====================================================
        #
        # Context
        #
        # = AttentionWeights × V
        context_vector = attn_weights @ v

        # Move heads back
        context_vector = context_vector.transpose(1, 2)

        # Ensure contiguous memory
        context_vector = context_vector.contiguous()

        # Merge heads
        #
        # (B, seq_len, d_model)
        context_vector = context_vector.view(
            batch_size,
            seq_len,
            self.d_model
        )

        # ====================================================
        # Step 9: Final Output Projection
        # ====================================================
        output = self.W_o(context_vector)

        return output


# ============================================================
# Testing the MLA Layer
# ============================================================

# Hidden dimension
d_model = 512

# Number of heads
num_heads = 8

# Latent dimension
#
# Must be smaller than d_model
#
# Example:
# 128 < 512
d_latent = 128

# Batch size
batch_size = 4

# Sequence length
seq_len = 64

# ============================================================
# MLA vs MHA vs MQA
# ============================================================
#
# MHA:
#
# Cache:
#     K1,V1
#     K2,V2
#     ...
#     K8,V8
#
# Memory Usage:
#     Highest
#
#
# ------------------------------------------------------------
#
# MQA:
#
# Cache:
#     Shared K
#     Shared V
#
# Memory Usage:
#     Lower
#
#
# ------------------------------------------------------------
#
# MLA (DeepSeek):
#
# Cache:
#     c_kv
#
# Where:
#
# c_kv = Compressed KV representation
#
# Example:
#
# MHA stores:
#     512 dimensions
#
# MLA stores:
#     128 dimensions
#
# Memory Reduction:
#
#     512 / 128 = 4× smaller
#
#
# ------------------------------------------------------------
#
# DeepSeek Core Idea:
#
# Input
#   ↓
# Compress KV
#   ↓
# Store c_kv in KV Cache
#   ↓
# Reconstruct K and V when needed
#   ↓
# Perform normal attention
#
# This is the fundamental innovation behind
# Multi-Head Latent Attention (MLA).
# ============================================================

# Create MLA layer
mla_layer = MultiHeadLatentAttention(
    d_model,
    num_heads,
    d_latent
)

# Create random input tensor
#
# Shape:
# (4, 64, 512)
dummy_input = torch.randn(
    batch_size,
    seq_len,
    d_model
)

# Run forward pass
output = mla_layer(dummy_input)

# Display results
print("MLA Layer successful!")

# Input tensor shape
print(f"Input shape: {dummy_input.shape}")

# Output tensor shape
print(f"Output shape: {output.shape}")


MLA Layer successful!
Input shape: torch.Size([4, 64, 512])
Output shape: torch.Size([4, 64, 512])


# Building the Fused MLA and Decoupled RoPE Module

In [2]:
# ============================================================
# DeepSeek Attention Architecture
# ============================================================
#
# Fused MLA + Decoupled RoPE:
#
# Purpose:
# - Implement DeepSeek's full attention mechanism
# - Combine MLA (Multi-Head Latent Attention)
# - Combine Decoupled RoPE (Rotary Positional Encoding)
# - Understand how DeepSeek handles content and position separately
#
# Key Innovation:
#
# Traditional Transformer:
#     Q, K contain BOTH content and position information
#
# DeepSeek:
#     Content information → MLA Path
#     Position information → RoPE Path
#
# Then:
#
# Final Attention Score =
#     Content Score + Position Score
#
# This is called Decoupled RoPE because positional
# information is separated from content information.
# ============================================================


# ============================================================
# Import Required Libraries
# ============================================================

import torch  # PyTorch tensor library
import torch.nn as nn  # Neural network layers
import math  # Mathematical functions


# ============================================================
# Rotary Positional Encoding (RoPE)
# ============================================================
#
# RoPE does NOT add position embeddings.
#
# Instead:
#
# It rotates Query and Key vectors in vector space.
#
# This allows the model to naturally learn relative
# token positions.
#
# Used in:
# - LLaMA
# - DeepSeek
# - Qwen
# - Mistral
# ============================================================

class RotaryPositionalEncoding(nn.Module):

    """
    Applies Rotary Positional Encoding (RoPE).

    Instead of:

        x + positional_embedding

    RoPE performs:

        x -> rotated(x)

    using complex number rotations.
    """

    def __init__(self, d_head, max_seq_len=2048):

        super().__init__()

        # ----------------------------------------------------
        # Create Theta Values
        # ----------------------------------------------------
        #
        # Formula:
        #
        # theta_i = 1 / 10000^(2i/d_head)
        #
        # Shape:
        # (d_head/2,)
        theta = 1.0 / (
            10000 **
            (
                torch.arange(
                    0,
                    d_head,
                    2
                ).float() / d_head
            )
        )

        # Store as model buffer
        self.register_buffer(
            'theta',
            theta
        )

        # ----------------------------------------------------
        # Create Position Indices
        # ----------------------------------------------------
        #
        # Example:
        #
        # [0]
        # [1]
        # [2]
        # ...
        positions = torch.arange(
            max_seq_len
        ).unsqueeze(1)

        # ----------------------------------------------------
        # Calculate Frequencies
        # ----------------------------------------------------
        #
        # freqs = position × theta
        #
        # Shape:
        # (max_seq_len, d_head/2)
        freqs = positions * self.theta.unsqueeze(0)

        # ----------------------------------------------------
        # Create Complex Rotation Matrix
        # ----------------------------------------------------
        #
        # Complex Number:
        #
        # cos(freq)
        # +
        # i*sin(freq)
        #
        # Equivalent to:
        #
        # e^(i*freq)
        #
        self.register_buffer(
            'freqs_cis',
            torch.polar(
                torch.ones_like(freqs),
                freqs
            )
        )

    def forward(self, x):

        # Input shape:
        #
        # (batch,
        #  num_heads,
        #  seq_len,
        #  d_head)
        seq_len = x.shape[2]

        # ----------------------------------------------------
        # Convert Real Tensor into Complex Tensor
        # ----------------------------------------------------
        #
        # Pair dimensions:
        #
        # [x1,x2]
        # [x3,x4]
        #
        # become:
        #
        # x1 + ix2
        # x3 + ix4
        x_complex = x.float().reshape(
            *x.shape[:-1],
            -1,
            2
        )

        # Convert into PyTorch complex datatype
        x_complex = torch.view_as_complex(
            x_complex
        )

        # ----------------------------------------------------
        # Select Required Frequencies
        # ----------------------------------------------------
        #
        # Shape:
        # (1,1,seq_len,d_head/2)
        freqs_cis = (
            self.freqs_cis[:seq_len, :]
            .unsqueeze(0)
            .unsqueeze(0)
        )

        # ----------------------------------------------------
        # Apply Rotation
        # ----------------------------------------------------
        #
        # Complex multiplication performs
        # the positional rotation.
        x_rotated = x_complex * freqs_cis

        # ----------------------------------------------------
        # Convert Back to Real Numbers
        # ----------------------------------------------------
        x_rotated = torch.view_as_real(
            x_rotated
        )

        # Restore original dimension
        x_rotated = x_rotated.flatten(3)

        return x_rotated.type_as(x)


# ============================================================
# DeepSeek Attention
# ============================================================
#
# Combines:
#
# 1. MLA Content Path
# 2. RoPE Position Path
#
# Final Score:
#
# Attention Score =
#     Content Score
#     +
#     Position Score
# ============================================================

class DeepSeekAttention(nn.Module):

    """
    Full DeepSeek Attention Layer.

    Architecture:

    Input
      |
      +---- Content Path (MLA)
      |
      +---- Position Path (RoPE)
      |
      Merge Scores
      |
      Softmax
      |
      Output
    """

    def __init__(
        self,
        d_model,
        num_heads,
        d_latent,
        d_rope,
        dropout=0.0,
        max_seq_len=2048
    ):

        super().__init__()

        # Ensure clean head splitting
        assert d_model % num_heads == 0, \
            "d_model must be divisible by num_heads"

        # Total hidden dimension
        self.d_model = d_model

        # Number of attention heads
        self.num_heads = num_heads

        # Dimension per head
        self.d_head = d_model // num_heads

        # MLA latent dimension
        self.d_latent = d_latent

        # RoPE dimension
        #
        # Used only for positional vectors
        self.d_rope = d_rope

        # ====================================================
        # A. Content Path (MLA)
        # ====================================================

        # Query projection
        self.W_q_content = nn.Linear(
            d_model,
            d_model
        )

        # Compress KV
        self.W_dkv_content = nn.Linear(
            d_model,
            d_latent
        )

        # Reconstruct Key
        self.W_uk_content = nn.Linear(
            d_latent,
            d_model
        )

        # Reconstruct Value
        self.W_uv_content = nn.Linear(
            d_latent,
            d_model
        )

        # ====================================================
        # B. Position Path (RoPE)
        # ====================================================

        # Positional Key projection
        self.W_k_pos = nn.Linear(
            d_model,
            d_rope * num_heads
        )

        # Positional Query projection
        self.W_q_pos = nn.Linear(
            d_model,
            d_rope * num_heads
        )

        # Rotary Positional Encoding module
        self.rope = RotaryPositionalEncoding(
            d_rope,
            max_seq_len
        )

        # ====================================================
        # C. Output Projection
        # ====================================================

        self.W_o = nn.Linear(
            d_model,
            d_model
        )

        # Dropout layer
        self.dropout = nn.Dropout(dropout)

        # Causal mask
        self.register_buffer(
            'mask',
            torch.triu(
                torch.ones(
                    1,
                    1,
                    max_seq_len,
                    max_seq_len
                ),
                diagonal=1
            ).bool()
        )

    def forward(self, x):

        # Input shape:
        # (batch_size, seq_len, d_model)
        batch_size, seq_len, _ = x.shape

        # ====================================================
        # A. Content Path
        # ====================================================

        # Content Query
        q_c = self.W_q_content(x)

        q_c = q_c.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_head
        ).transpose(1, 2)

        # ----------------------------------------------------
        # Compress KV Information
        # ----------------------------------------------------
        #
        # MLA Cache:
        #
        # Only this latent tensor is cached.
        c_kv = self.W_dkv_content(x)

        # Reconstruct Key
        k_c = self.W_uk_content(c_kv)

        k_c = k_c.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_head
        ).transpose(1, 2)

        # Reconstruct Value
        v_c = self.W_uv_content(c_kv)

        v_c = v_c.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_head
        ).transpose(1, 2)

        # ====================================================
        # B. Position Path
        # ====================================================

        # Position Query
        q_r_unrotated = self.W_q_pos(x)

        q_r_unrotated = q_r_unrotated.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_rope
        ).transpose(1, 2)

        # Position Key
        k_r_unrotated = self.W_k_pos(x)

        k_r_unrotated = k_r_unrotated.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_rope
        ).transpose(1, 2)

        # ----------------------------------------------------
        # Apply RoPE Rotation
        # ----------------------------------------------------

        q_r = self.rope(
            q_r_unrotated
        )

        k_r = self.rope(
            k_r_unrotated
        )

        # ====================================================
        # C. Compute Attention Scores
        # ====================================================

        # Content similarity score
        content_scores = (
            q_c @ k_c.transpose(-2, -1)
        ) / (self.d_head ** 0.5)

        # Positional similarity score
        position_scores = (
            q_r @ k_r.transpose(-2, -1)
        ) / (self.d_rope ** 0.5)

        # Final DeepSeek score
        #
        # Content + Position
        attn_scores = (
            content_scores
            +
            position_scores
        )

        # ====================================================
        # D. Mask Future Tokens
        # ====================================================

        attn_scores = attn_scores.masked_fill(
            self.mask[:, :, :seq_len, :seq_len],
            float('-inf')
        )

        # ====================================================
        # E. Softmax
        # ====================================================

        attn_weights = torch.softmax(
            attn_scores,
            dim=-1
        )

        attn_weights = self.dropout(
            attn_weights
        )

        # ====================================================
        # F. Compute Context Vector
        # ====================================================
        #
        # Only content value matrix is used.
        #
        # Position information already influenced
        # attention scores.
        context_vector = (
            attn_weights @ v_c
        )

        context_vector = (
            context_vector
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                seq_len,
                self.d_model
            )
        )

        # ====================================================
        # G. Final Projection
        # ====================================================

        output = self.W_o(
            context_vector
        )

        return output


# ============================================================
# Testing DeepSeek Attention Layer
# ============================================================

# Hidden dimension
d_model = 512

# Number of heads
num_heads = 8

# MLA latent dimension
d_latent = 128

# RoPE dimension
#
# Usually d_head or smaller
d_rope = 64

# Batch size
batch_size = 4

# Sequence length
seq_len = 64

# Create DeepSeek Attention Layer
deepseek_attn_layer = DeepSeekAttention(
    d_model,
    num_heads,
    d_latent,
    d_rope
)

# Random input tensor
#
# Shape:
# (4,64,512)
dummy_input = torch.randn(
    batch_size,
    seq_len,
    d_model
)

# Forward pass
output = deepseek_attn_layer(
    dummy_input
)

print("DeepSeekAttention Layer successful!")

print(f"Input shape: {dummy_input.shape}")

print(f"Output shape: {output.shape}")



DeepSeekAttention Layer successful!
Input shape: torch.Size([4, 64, 512])
Output shape: torch.Size([4, 64, 512])
